<a href="https://colab.research.google.com/github/muneer-ahmad10/Natural-Language-processing/blob/main/Project_02_Mini_GPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim

In [5]:
#Training Data

text = """
i love machine learning
machine learning is amazing
deep learning uses neural networks
transformers changed ai
i love transformers
ai is the future
"""

In [6]:
words=text.split()
vocab=sorted(set(words))
# vocab=sorted(set(text.split()))

# Create Word -> Number
word_to_idx={
    word : i
    for i ,word in enumerate(vocab)
}
word_to_idx



{'ai': 0,
 'amazing': 1,
 'changed': 2,
 'deep': 3,
 'future': 4,
 'i': 5,
 'is': 6,
 'learning': 7,
 'love': 8,
 'machine': 9,
 'networks': 10,
 'neural': 11,
 'the': 12,
 'transformers': 13,
 'uses': 14}

In [7]:
#Reverse the Dictionary later : number -> Word During generation
idx_to_word={
    i : word
    for  word,i in word_to_idx.items()
}
idx_to_word

{0: 'ai',
 1: 'amazing',
 2: 'changed',
 3: 'deep',
 4: 'future',
 5: 'i',
 6: 'is',
 7: 'learning',
 8: 'love',
 9: 'machine',
 10: 'networks',
 11: 'neural',
 12: 'the',
 13: 'transformers',
 14: 'uses'}

In [8]:
enumerate(vocab)

In [9]:
 word_to_idx

{'ai': 0,
 'amazing': 1,
 'changed': 2,
 'deep': 3,
 'future': 4,
 'i': 5,
 'is': 6,
 'learning': 7,
 'love': 8,
 'machine': 9,
 'networks': 10,
 'neural': 11,
 'the': 12,
 'transformers': 13,
 'uses': 14}

In [10]:
idx_to_word

{0: 'ai',
 1: 'amazing',
 2: 'changed',
 3: 'deep',
 4: 'future',
 5: 'i',
 6: 'is',
 7: 'learning',
 8: 'love',
 9: 'machine',
 10: 'networks',
 11: 'neural',
 12: 'the',
 13: 'transformers',
 14: 'uses'}

In [11]:
sequence_length = 3

inputs = []
targets = []

tokenized = [
    word_to_idx[word]
    for word in words
]

In [12]:
len(tokenized)

23

In [13]:


for i in range(len(tokenized)-sequence_length):

    input_seq = tokenized[i:i+sequence_length]

    target = tokenized[i+sequence_length]

    inputs.append(input_seq)

    targets.append(target)

inputs = torch.tensor(inputs)
targets = torch.tensor(targets)

In [14]:
# class MiniGPT(nn.Module):

#     def __init__(
#         self,
#         vocab_size,
#         embed_dim,
#         num_heads,
#         hidden_dim
#     ):

#         super().__init__()

#         # embeddings
#         self.embedding = nn.Embedding(
#             vocab_size,
#             embed_dim
#         )

#         # attention
#         self.attention = nn.MultiheadAttention(
#             embed_dim,
#             num_heads,
#             batch_first=True
#         )

#         # feed forward
#         self.ff = nn.Sequential(
#             nn.Linear(embed_dim, hidden_dim),
#             nn.ReLU(),
#             nn.Linear(hidden_dim, embed_dim)
#         )

#         # final prediction
#         self.fc = nn.Linear(
#             embed_dim,
#             vocab_size
#         )

#     def forward(self, x):

#         # embeddings
#         x = self.embedding(x)

#         # self attention
#         attn_output, _ = self.attention(
#             x,
#             x,
#             x
#         )

#         # feed forward
#         x = self.ff(attn_output)

#         # take last token
#         x = x[:, -1, :]

#         # predict next word
#         out = self.fc(x)

#         return out

class MiniGPT(nn.Module):
  def __init__(self, vocab_size,embed_dim,num_heads,hidden_dim):
     super().__init__()

     #embedding
     self.embedding=nn.Embedding(
         vocab_size,
         embed_dim
     )

     self.attention=nn.MultiheadAttention(
         embed_dim,
         num_heads,
         batch_first=True
     )

     self.ff=nn.Sequential(
         nn.Linear(embed_dim,hidden_dim),
         nn.ReLU(),
         nn.Linear(hidden_dim,embed_dim)
     )
     self.fc=nn.Linear(
         embed_dim,
         vocab_size
     )

  def forward(self,x):
    x=self.embedding(x)

    attn_output, _=self.attention(x,x,x)

    x=self.ff(attn_output)


    x = x[:, -1, :]

    out=self.fc(x)

    return out



In [15]:
model = MiniGPT(
    vocab_size=len(vocab),
    embed_dim=32,
    num_heads=2,
    hidden_dim=64
)

In [16]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.01
)

In [17]:
for epoch in range(300):

    optimizer.zero_grad()

    outputs = model(inputs)

    loss = criterion(
        outputs,
        targets
    )

    loss.backward()

    optimizer.step()

    if epoch % 50 == 0:
        print(
            f"Epoch {epoch}, Loss: {loss.item()}"
        )

Epoch 0, Loss: 2.710062265396118
Epoch 50, Loss: 2.5510325940558687e-06
Epoch 100, Loss: 2.980230249249871e-07
Epoch 150, Loss: 2.086161714487389e-07
Epoch 200, Loss: 1.6093250110316148e-07
Epoch 250, Loss: 1.3113017871546617e-07


In [18]:
def generate_text(start_text, length=5):

    model.eval()

    words_input = start_text.split()

    for _ in range(length):

        input_seq = [
            word_to_idx[word]
            for word in words_input[-3:]
        ]

        input_tensor = torch.tensor(
            [input_seq]
        )

        with torch.no_grad():

            output = model(input_tensor)

            predicted_idx = torch.argmax(
                output,
                dim=1
            ).item()

        predicted_word = idx_to_word[
            predicted_idx
        ]

        words_input.append(predicted_word)

    return " ".join(words_input)

In [19]:
print(
    generate_text("i love")
)

i love future transformers changed ai i


In [20]:
print(
    generate_text("deep learning")
)

deep learning neural networks networks transformers changed


In [21]:
print(
    generate_text("amazing")
)

amazing learning learning learning networks networks


## Now upgrading to GPT


*   Adding POSITIONAL ENCODING + CAUSAL MASKING




In [22]:
class BetterGPT(nn.Module):
  def __init__(self,vocab_size,embed_dim,num_heads,hidden_dim,max_len=100):
      super().__init__()

      self.embedding=nn.Embedding(
          vocab_size,
          embed_dim
      )

      self.position_embedding=nn.Embedding(
          max_len,
          embed_dim
      )

      self.attn=nn.MultiheadAttention(
        embed_dim,
        num_heads,
        batch_first=True
      )

      self.ff=nn.Sequential(
          nn.Linear(embed_dim,hidden_dim),
          nn.ReLU(),
          nn.Linear(hidden_dim,embed_dim)
      )

      self.fc=nn.Linear(
          embed_dim,
          vocab_size
      )
  def feed_forward(self,x):
      seq_len=x.size(1)

      positions=torch.arange(0,seq_len).unsqueeze(0)

      x=self.embedding(x) + self.position_embedding(positions)

      mask=torch.triu(torch.ones(seq_len,seq_len),diagonal=1).bool()

      attn_output, _ = self.attention(
            x,
            x,
            x,
            attn_mask=mask
        )

      x = self.ff(attn_output)

      x = x[:, -1, :]

      out = self.fc(x)

      return out







In [23]:
better_model = MiniGPT(
    vocab_size=len(vocab),
    embed_dim=32,
    num_heads=2,
    hidden_dim=64
)

In [24]:
criterion_i = nn.CrossEntropyLoss()

optimizer_i = optim.Adam(
    model.parameters(),
    lr=0.01
)

In [25]:
for epoch in range(300):

    optimizer_i.zero_grad()

    outputs = model(inputs)

    loss = criterion_i(
        outputs,
        targets
    )

    loss.backward()

    optimizer_i.step()

    if epoch % 50 == 0:
        print(
            f"Epoch {epoch}, Loss: {loss.item()}"
        )

Epoch 0, Loss: 1.0132787053862558e-07
Epoch 50, Loss: 0.0
Epoch 100, Loss: 0.0
Epoch 150, Loss: 0.0
Epoch 200, Loss: 0.0
Epoch 250, Loss: 0.0


In [26]:
def generate_text_i(start_text, length=5):

    better_model.eval()

    words_input_i = start_text.split()

    for _ in range(length):

        input_seq = [
            word_to_idx[word]
            for word in words_input_i[-3:]
        ]

        input_tensor = torch.tensor(
            [input_seq]
        )

        with torch.no_grad():

            output = better_model(input_tensor)

            predicted_idx = torch.argmax(
                output,
                dim=1
            ).item()

        predicted_word = idx_to_word[
            predicted_idx
        ]

        words_input_i.append(predicted_word)

    return " ".join(words_input_i)

In [27]:
print(
    generate_text_i("networks")
)

networks future future future the the


In [35]:
print(
    generate_text_i("i love machine")
)

i love machine the future the the the


In [29]:
#updated generate text function
def generate_text_x(start_text, length=5):

    better_model.eval()

    words_input = start_text.split()

    for _ in range(length):

        input_seq = [
            word_to_idx[word]
            for word in words_input[-3:]
        ]

        input_tensor = torch.tensor(
            [input_seq]
        )

        with torch.no_grad():

            output = better_model(input_tensor)

            # temperature
            output = output / 0.8

            # probabilities
            probs = torch.softmax(
                output,
                dim=1
            )

            # sampling
            predicted_idx = torch.multinomial(
                probs,
                num_samples=1
            ).item()

        predicted_word = idx_to_word[
            predicted_idx
        ]

        words_input.append(predicted_word)

    return " ".join(words_input)

In [34]:
print(
    generate_text_x("i love machine")
)

i love machine networks networks i changed the


**Practice**

In [31]:
import torch

# Create a tensor with shape: (Batch=32, Channels=3, Height=28, Width=28)
x = torch.randn(32, 3, 28, 28)

# Get the number of channels using x.size(1)
in_channels = x.size(3)
print(in_channels)  # Output: 3
print(type(in_channels))

28
<class 'int'>


In [32]:
x.shape

torch.Size([32, 3, 28, 28])

In [33]:
torch.triu(torch.ones(32,32),diagonal=1).bool()

tensor([[False,  True,  True,  ...,  True,  True,  True],
        [False, False,  True,  ...,  True,  True,  True],
        [False, False, False,  ...,  True,  True,  True],
        ...,
        [False, False, False,  ..., False,  True,  True],
        [False, False, False,  ..., False, False,  True],
        [False, False, False,  ..., False, False, False]])

**“Every architecture in NLP evolved to solve limitations of the previous** **architecture.”**

**That single sentence shows:**

**🚀 deep understanding**